<a href="https://colab.research.google.com/github/thomas-sutter/ai-aml-mvp2value/blob/feat%2Feda-nb2-modelling-2025-v0.2/2_modelling_baselines_and_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## STEP 0 — Environment & Libraries (Run Card)

**Goal**  
Initialize a reproducible modeling environment and load core libraries. Optional packages (XGBoost, SHAP, imbalanced-learn, LightGBM) are imported on a best-effort basis without blocking the baseline flow.

**Why**  
A clean setup prevents runtime breaks and guarantees traceability (fixed seeds, consistent package set, stable pipelines). Reviewers can reliably reproduce results.

**What this cell does**
- Imports: `numpy`, `pandas`, `scikit-learn` (pipelines, preprocessing, metrics), `RandomForest`, `LogisticRegression`, calibration.
- Tries optional imports: `xgboost`, `shap`; sets flags `HAS_XGB`, `HAS_SHAP`.
- Sets a global random seed `RNG = 42` and suppresses warnings for clean output.

**How**
1. Perform mandatory imports.  
2. Wrap optional imports in `try/except` and expose boolean flags for downstream logic.  
3. Define `RNG` and call `np.random.seed(RNG)` for deterministic splits/CV.

**Inputs / Dependencies**  
Python ≥ 3.10 recommended; `scikit-learn` ≥ 1.2. No data access in this step.

**Outputs / Side-effects**  
No artifacts written. Only module namespaces, feature flags, and global seed are set in the kernel state.

**Acceptance criteria**
- Cell executes without exceptions.  
- Flags reflect package availability (`True/False`).  
- No noisy warnings in the output.

**Risks & controls**
- *Version drift*: In **STEP 1**, log package versions (to README/experiment log) to ensure reproducibility.  
- *Silent fallback*: If `HAS_XGB=False` or `HAS_SHAP=False`, proceed with LR/RF baselines; enable advanced models only when installed.

**Next step**  
STEP 1 — Load data & define target/features; then time-based split and baseline metrics (PR-AUC first).


In [1]:
# ==== CELL 0: Setup (install optional libs) ====
# If you want SHAP or LightGBM, uncomment:
# !pip -q install shap lightgbm imbalanced-learn

import os, re, math, warnings, numpy as np, pandas as pd
from dataclasses import dataclass
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score, precision_recall_curve, confusion_matrix,
    precision_score, recall_score, f1_score, roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV

# Optional imports (graceful fallback)
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

RNG = 42
np.random.seed(RNG)


## STEP 1 — Environment & Paths (Colab-ready)

**Goal**  
Provide a portable, reproducible setup that aligns paths with Notebook 1, works both in Google Colab and locally, and declares the candidate data files (Parquet preferred, CSV as fallback).

**Why**  
Centralized path management and a fixed RNG seed reduce “it works on my machine” issues, keep NB1↔NB2 consistent, and make runs reproducible for reviewers (professor, coach, future employers).

**What this cell does**  
- Imports essentials (`numpy`, `pandas`, `os`, etc.) and suppresses warnings.  
- *Optionally* mounts Google Drive when running in Colab.  
- Defines `BASE_PATH` identical to Notebook 1’s data directory.  
- Sets `RNG = 42` and applies `np.random.seed`.  
- Declares candidate inputs with clear precedence:  
  1) `xfers_structured_clean_v1.parquet` (primary)  
  2) `xfers_structured_clean_v1.csv`  
  3) `xfers_enriched_v3.csv`, `xfers_enriched_v2.csv` (legacy)  
- Prints a quick existence audit for each candidate.

**How**  
1. Use a `try/except` guard for `google.colab` imports to keep local runs unaffected.  
2. Keep `BASE_PATH` in one place to minimize edits across notebooks.  
3. Prefer Parquet (schema fidelity, speed); retain CSV for portability.

**Inputs / Dependencies**  
- Data artifacts produced by Notebook 1 under `BASE_PATH`.  
- Colab users need Drive access; local users need the same folder layout (or override `BASE_PATH`).

**Outputs / Side-effects**  
- Kernel state now includes `BASE_PATH`, `RNG`, and paths to candidate datasets.  
- Console prints: base path and file-existence flags.

**Acceptance criteria**  
- Cell executes without errors both in Colab and locally.  
- At least one candidate data file exists (ideally the Parquet).  
- RNG is fixed at 42 and warnings are suppressed.

**Risks & controls**  
- *Path drift*: If your directory changes, update **only** `BASE_PATH`. Optionally support an env var override:
  ```python
  BASE_PATH = os.getenv("AML_DATA_PATH", BASE_PATH)


In [2]:
# ==== Cell 1 (robust): Environment & Paths ====
import os, warnings, numpy as np, pandas as pd
from glob import glob
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

PROJECT_ROOT = "/content/drive/MyDrive/Portfolio/AML/core_banking_and_money_laundering"

# Verzeichnisse, die durchsucht werden (Priorität von links nach rechts)
SEARCH_DIRS = [
    f"{PROJECT_ROOT}/out",
    f"{PROJECT_ROOT}/data",
    f"{PROJECT_ROOT}",
]

# Dateinamen-/Muster in Prioritätsreihenfolge
CANDIDATE_PATTERNS = [
    "xfers_structured_clean_v1.parquet",
    "xfers_structured_clean_v*.parquet",
    "xfers_structured_clean_v1.csv",
    "xfers_structured_clean_v1.csv.gz",
    "xfers_enriched_v3.csv",
    "xfers_enriched_v2.csv",
]

print("PROJECT_ROOT:", PROJECT_ROOT)
for d in SEARCH_DIRS:
    print("Dir exists:", os.path.exists(d), d)


Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/Portfolio/AML/core_banking_and_money_laundering
Dir exists: True /content/drive/MyDrive/Portfolio/AML/core_banking_and_money_laundering/out
Dir exists: True /content/drive/MyDrive/Portfolio/AML/core_banking_and_money_laundering/data
Dir exists: True /content/drive/MyDrive/Portfolio/AML/core_banking_and_money_laundering


## STEP 2 — Load Dataset (typed & robust, with fallbacks)

**Goal**  
Load the modeling snapshot produced in Notebook 1 using a clear priority order (Parquet → clean CSV → enriched CSV v3/v2), keeping the flow resilient across environments.

**Why**  
A deterministic load order avoids “file roulette,” Parquet preserves schema and speeds I/O, and CSV fallbacks keep the notebook runnable if Parquet is missing. This improves reproducibility for reviewers and future employers.

**What this cell does**  
- Implements a **waterfall loader**:
  1) `xfers_structured_clean_v1.parquet` (primary, dtype-safe & fast)  
  2) `xfers_structured_clean_v1.csv` (clean CSV fallback)  
  3) `xfers_enriched_v3.csv` / `xfers_enriched_v2.csv` (last-resort inputs)  
- Returns the dataframe and a `load_src` tag (`"parquet" | "csv_clean" | "enriched_v3" | "enriched_v2"`).  
- Prints the source and shape for quick auditing; previews a couple of rows.

**How**  
1. Check existence in priority order with `os.path.exists`.  
2. Read Parquet via `pd.read_parquet`; read CSV with `pd.read_csv(low_memory=False)`; for enriched files, parse candidate timestamp `Txn_TS` if present.  
3. Keep the loader **pure** (no mutation other than read) and defer typing/normalization (e.g., timestamp enforcement) to subsequent steps.

**Inputs / Dependencies**  
- Artifacts written by Notebook 1 under `BASE_PATH`.  
- `pandas` with Parquet support (e.g., `pyarrow`) if Parquet path is used.

**Outputs / Side-effects**  
- Variables: `df`, `load_src`.  
- Console audit: `Loaded from: <source> | shape=(rows, cols)` and a small preview.

**Acceptance criteria**  
- Cell executes without errors and sets `load_src ∈ {"parquet","csv_clean","enriched_v3","enriched_v2"}`.  
- `df.shape[0] > 0` (non-empty


In [3]:
# ==== Cell 2 (robust): Locate & Load dataset ====
def locate_dataset(search_dirs, patterns, deep_search=True, max_hits=20):
    hits = []
    # 1) direkte Suche in SEARCH_DIRS
    for d in search_dirs:
        if not os.path.exists(d):
            continue
        for patt in patterns:
            hits.extend(glob(os.path.join(d, patt)))
    # 2) optional: tiefe Suche im Projektordner
    if not hits and deep_search and os.path.exists(PROJECT_ROOT):
        deep_patterns = [
            "**/xfers_structured_clean*.parquet",
            "**/xfers_structured_clean*.csv*",
            "**/xfers_enriched_v*.csv",
        ]
        for patt in deep_patterns:
            hits.extend(glob(os.path.join(PROJECT_ROOT, patt), recursive=True))
            if len(hits) >= max_hits:
                break
    if not hits:
        return None
    # Neuste Datei zuerst
    hits = sorted(hits, key=lambda p: os.path.getmtime(p), reverse=True)
    return hits[0]

DATA_PATH = locate_dataset(SEARCH_DIRS, CANDIDATE_PATTERNS)
if not DATA_PATH:
    raise FileNotFoundError(
        "No suitable input found.\n"
        "Fix: (A) Run NB1 export to write /out/xfers_structured_clean_v1.parquet, or "
        "(B) set DATA_PATH manually to your file."
    )

print("Using DATA_PATH:", DATA_PATH)
ext = os.path.splitext(DATA_PATH)[1].lower()

# Laden (Zeitspalte wird in Cell 3 geparst → hier ohne parse_dates laden)
if ext == ".parquet":
    df = pd.read_parquet(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH, low_memory=False)

print("Loaded shape:", df.shape)
df.head(2)


Using DATA_PATH: /content/drive/MyDrive/Portfolio/AML/core_banking_and_money_laundering/data/xfers_structured_clean_v1.parquet
Loaded shape: (1082800, 96)


,Timestamp,From_Bank,From_Account,To_Bank,To_Account,Amount_Paid,Payment_Currency,Amount_Received,Receiving_Currency,Payment_Format,...,in_sum_30d,bp_cnt_30d,bp_sum_30d,out_avg_7d,in_avg_7d,ct_ratio_out_in_7d,out_avg_30d,in_avg_30d,ct_ratio_out_in_30d,Is_RoundTrip_72h
0,2024-01-20 12:27:57.600,Lilian's Detroit Lakes Thrift,805391059,Citi,335544450,1516.01,USD,1516.01,USD,Debit Non-Prepaid,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,2024-09-26 14:40:27.200,Jaelynn's Indianapolis Bank,2080463420,Elias's Sanford Savings Bank,469866432,195.17,USD,195.17,USD,Debit Non-Prepaid,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


## STEP 3 — Target & Event Time (auto-detect, locked to NB1 defaults)

**Goal**  
Confirm the target and event-time columns, enforce canonical types (binary target, UTC `datetime64[ns]`), and establish strict chronological ordering—aligned with Notebook 1 conventions.

**Why**  
Time-aware modeling in AML must respect real-world chronology to avoid leakage. Locking to NB1’s defaults (`Is_Money_Laundering`, `Txn_TS`) keeps both notebooks consistent and reproducible, while still allowing auto-detection if schemas vary.

**What this cell does**  
- **Detects** the target column from curated candidates; **prioritizes** `Is_Money_Laundering`.  
- **Locks** the time column to `Txn_TS` when present; otherwise falls back to known time candidates.  
- **Normalizes** time to UTC `datetime64[ns]`, **drops** records with missing target/time, **sorts ascending**, resets index.  
- **Coerces** the target to `{0,1}` if it arrives as string/bool (maps `'1'|'true'|'yes' → 1`).  
- **Audits** and prints: target, time, positive rate, row count, and time span.

**How**  
1. Resolve `target_col` from `TARGET_CANDIDATES`.  
2. Resolve `time_col` as `'Txn_TS'` if available; else scan `TIME_CANDIDATES`.  
3. `pd.to_datetime(..., utc=True).astype('datetime64[ns]')` then `dropna`, `sort_values(time_col)`, `reset_index`.  
4. If target not numeric/bool, coerce via string membership to `{0,1}`.  
5. Compute and print the positive-class rate (`mean`).

**Inputs / Dependencies**  
- `df` loaded in prior step, with NB1-compatible schema if available.  
- `pandas` for datetime parsing; NB1 produced `Txn_TS` and `Is_Money_Laundering` ideally.

**Outputs / Side-effects**  
- Mutated `df` with enforced time dtype, chronological ordering, and binary target.  
- Variables in scope: `t


In [4]:
# ==== CELL 3: Target & Time columns (tz-safe, NB1-aligned) ====
"""
Purpose:
- Detect target and time columns (NB1 defaults: target='Is_Money_Laundering', time='Txn_TS').
- Normalize time to naive-UTC (datetime64[ns]) to avoid tz issues in quantiles/splits.
- Ensure binary target (0/1).
"""

import numpy as np
import pandas as pd

# 1) Detect columns (prefer NB1 defaults)
TARGET_CANDIDATES = ["Is_Money_Laundering","is_money_laundering","Is_Laundering","label","Label","AML_Flag"]
TIME_CANDIDATES   = ["Txn_TS","Transfer_Timestamp","Txn_Timestamp","Transaction_Timestamp",
                     "Posted_Timestamp","Posted_Datetime","Event_Time","Timestamp","Date","Datetime"]

target_col = next((c for c in TARGET_CANDIDATES if c in df.columns), None)
time_col   = "Txn_TS" if "Txn_TS" in df.columns else next((c for c in TIME_CANDIDATES if c in df.columns), None)

assert target_col is not None, f"Target col not found among {TARGET_CANDIDATES}."
assert time_col   is not None, f"Time col not found among {TIME_CANDIDATES}."

# 2) Robust time normalizer → naive UTC
def to_naive_utc(series: pd.Series) -> pd.Series:
    # First try: make it tz-aware UTC, then drop tz → naive UTC
    s = pd.to_datetime(series, errors="coerce", utc=True)
    try:
        # If tz-aware, convert to UTC and drop timezone
        return s.dt.tz_convert("UTC").dt.tz_localize(None)
    except Exception:
        # Fallback: if tz-naive sneaks in, localize to UTC then drop tz
        s2 = pd.to_datetime(series, errors="coerce")
        try:
            s2 = s2.dt.tz_localize("UTC").dt.tz_convert("UTC").dt.tz_localize(None)
            return s2
        except Exception:
            # Last resort: return whatever to_datetime produced (may already be naive)
            return pd.to_datetime(series, errors="coerce")

df[time_col] = to_naive_utc(df[time_col])

# 3) Drop NA in key cols, sort by time
df = df.dropna(subset=[time_col, target_col]).sort_values(time_col).reset_index(drop=True)

# 4) Ensure binary target (0/1) without surprises
if pd.api.types.is_integer_dtype(df[target_col]) or pd.api.types.is_bool_dtype(df[target_col]):
    df[target_col] = (df[target_col].astype(int) > 0).astype(int)
else:
    # Map common truthy/falsey strings; unknowns default to 0
    mapping = {"1":1,"true":1,"yes":1,"y":1,"t":1,"ja":1,
               "0":0,"false":0,"no":0,"n":0,"f":0,"nein":0}
    df[target_col] = (
        df[target_col].astype(str).str.strip().str.lower().map(mapping).fillna(0).astype(int)
    )

# 5) Quick info
pos_rate = df[target_col].mean()
print(f"Target='{target_col}' | Time='{time_col}' | dtype(time)={df[time_col].dtype}")
print(f"Rows={len(df)} | PosRate={pos_rate:.4f} | Span={df[time_col].min()} → {df[time_col].max()}")


Target='Is_Laundering' | Time='Txn_TS' | dtype(time)=datetime64[ns]
Rows=1082800 | PosRate=0.0011 | Span=2024-01-20 12:27:57.600000 → 2025-04-02 13:39:01.800000


## STEP 4 — Feature Space (leakage-safe; numeric + low-card categoricals)

**Goal**  
Construct `X` and `y` with features that are safe for modeling: exclude IDs/PII and keep only numeric features plus low-cardinality categoricals suitable for one-hot encoding.

**Why**  
Explicit identifiers (account IDs, names, emails, etc.) and high-cardinality categoricals can leak identity or memorize patterns, inflating metrics and harming generalization—especially in AML where entities reappear over time.

**What this cell does**  
- Builds `y = df[target_col].values`.  
- Uses regex heuristics to detect and **drop ID/PII-like columns** (e.g., `*_ID`, `Name`, `Email`, `Phone`).  
- Removes meta columns (`target_col`, `time_col`).  
- Splits remaining columns into:
  - `num_cols`: numeric dtypes (kept as-is).  
  - `cat_cols`: non-numeric with **≤ 50 unique** values (kept for one-hot).  
- Creates feature frame `Xc` (meta + ID-like columns removed) and prints a summary.

**How**  
1. Define a curated set of **regex patterns** to catch IDs/PII and side-channel identifiers.  
2. Drop `{target_col, time_col}` and any column matching the patterns.  
3. Classify columns by dtype; enforce a **low-cardinality threshold** (`≤ 50`) for categoricals to avoid dimensionality blow-ups in OHE.  
4. Keep lists `num_cols`, `cat_cols` for downstream preprocessing pipelines.

**Inputs / Dependencies**  
- Cleaned and time-sorted `df` from Step 3.  
- Python `re`, `pandas` type checks.

**Outputs / Side-effects**  
- `Xc`: candidate feature frame (leakage-aware).  
- Lists: `num_cols`, `cat_cols`.  
- Console audit: counts of kept numeric/categorical vs dropped ID-like columns.

**Acceptance criteria**  
- `Xc` contains **no** `target_col` or `time_col`.  
- **Zero** columns with obvious identifiers (IDs, names, contact fields).  
- `cat_cols` all satisfy `nunique ≤ 50`.  
- `num_cols + cat_cols` spans all remaining columns of `Xc` (no unclassified leftovers).

**Risks & controls**  
- *False negatives in ID detection*: Extend regex list if domain columns slip through (e.g., `IBAN`, `SWIFT`, `KontoNr`).  
- *False positives (over-dropping)*: Review `id_like_cols` when drops are unexpectedly high; whitelist essential engineered features if needed.  
- *High-cardinal categoricals*: If a key feature is excluded due to cardinality, consider **hashing**, **WOE/target encoding with proper CV**, or **frequency binning** (but guard against leakage).  
- *PII compliance*: Ensure that any PII never reaches model inputs or artifacts.

**Observability (quick checks)**
```python
print("Dropped ID-like cols:", sorted(id


In [5]:
# ==== CELL 4: Feature space (drop IDs/Names; keep safe low-card categoricals) ====
"""
Purpose: Build X,y with leakage-safe features.
Rules:
- Drop explicit IDs / names / PII-like columns.
- Keep numeric features as-is; keep only low-card categoricals (<= 50 unique).
"""
y = df[target_col].values

# Heuristics for ID-like columns
ID_LIKE_PATTERNS = [
    r"\b.*_ID\b", r"(^ID$)", r"\bID\b",
    r"\b(Account|Acct|Customer|Client|User|Party|Bank|Card|Node|Edge).*(_ID|Number|No)\b",
    r"(^From_.*ID$)|(^To_.*ID$)", r"(.*Name$|.*Address$|.*Email$|.*Phone$|.*SSN$)"
]
meta_cols = {target_col, time_col}
id_like_cols = set()
for col in df.columns:
    if col in meta_cols:
        continue
    for pat in ID_LIKE_PATTERNS:
        if re.search(pat, col, flags=re.IGNORECASE):
            id_like_cols.add(col); break

Xc = df.drop(columns=list(meta_cols | id_like_cols))

num_cols, cat_cols = [], []
for c in Xc.columns:
    if pd.api.types.is_numeric_dtype(Xc[c]):
        num_cols.append(c)
    else:
        # Only include low-card categoricals to avoid blow-ups
        if Xc[c].nunique(dropna=True) <= 50:
            cat_cols.append(c)

print(f"Kept features: {len(num_cols)} numeric + {len(cat_cols)} categorical; Dropped {len(id_like_cols)} ID-like cols.")


Kept features: 52 numeric + 15 categorical; Dropped 20 ID-like cols.


## STEP 5 — Time-based split (quantile cutoffs: t80, t90)

**Goal**  
Create leakage-safe train/validation/test sets that respect chronology: **Train ≤ t80**, **Val (t80, t90]**, **Test > t90**.

**Why**  
In AML, future information must never influence past training. Time-ordered splits mimic production and reveal drift between periods.

**What this cell does**  
- Computes timestamp cutoffs at the 80th and 90th percentiles.  
- Builds boolean masks for train/val/test in **forward time**.  
- Partitions `Xc`/`y` accordingly and prints shapes and cutoffs.

**How**  
1. `t80 = quantile(0.80)`, `t90 = quantile(0.90)` on `df[time_col]`.  
2. Masks: `<= t80`, `(t80, t90]`, `> t90`.  
3. Index with `.loc[...]` to keep row alignment.

**Inputs / Dependencies**  
- `df` sorted by `time_col` (from STEP 3).  
- `Xc`, `y`, `time_col` defined.

**Outputs / Side-effects**  
- `X_train, y_train`, `X_val, y_val`, `X_test, y_test`.  
- Printed audit of shapes and date cutoffs.

**Acceptance criteria**  
- All three splits are **non-empty** and **disjoint**.  
- Time ranges are contiguous and ordered (no leakage).  
- Class prevalence is reported later per split (sanity check for drift).

**Risks & controls**  
- *Irregular time density / ties at cutoffs*: If Val/Test too small, adjust to (70/15/15) or use fixed calendar cutoffs.  
- *Entity leakage across time*: If the same account appears across splits with near-simultaneous events, consider **purging/blackout windows** or group-aware forward chaining.  
- *Imbalanced rare events*: Verify positive rates per split; re-balance or re-weight on Train only.

**Observability (quick checks)**
```python
# No overlap
assert not set(X_train.index) & set(X_val.index)
assert not set(X_train.index) & set(X_test.index)
assert not set(X_val.index)   & set(X_test.index)

# Split sizes
for name, X_, y_ in [("Train", X_train, y_train), ("Val", X_val, y_val), ("Test", X_test, y_test)]:
    print(name, "| n:", len(X_), "| pos rate:", round(float(np.mean(y_)), 4))

# Ranges
print("Train span:", X_train.index.min(), "→", X_train.index.max())
print("Val   span:", X_val.index.min(),   "→", X_val.index.max())
print("Test  span:", X_test.index.min(),  "→", X_test.index.max())


In [6]:
# ==== CELL 5: Time-based split (Train ≤ t1, Val (t1,t2], Test > t2) ====
"""
Purpose: Avoid leakage by splitting on time (not random).
Defaults: 80% train, 10% val, 10% test by timestamps.
"""
t80 = df[time_col].quantile(0.80)
t90 = df[time_col].quantile(0.90)

train_idx = df[time_col] <= t80
val_idx   = (df[time_col] > t80) & (df[time_col] <= t90)
test_idx  = df[time_col] > t90

X_train, y_train = Xc.loc[train_idx], y[train_idx]
X_val,   y_val   = Xc.loc[val_idx],   y[val_idx]
X_test,  y_test  = Xc.loc[test_idx],  y[test_idx]

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print("Cutoffs:", df[time_col].min(), "→", t80, "→", t90, "→", df[time_col].max())


Train: (866240, 74) | Val: (108280, 74) | Test: (108280, 74)
Cutoffs: 2024-01-20 12:27:57.600000 → 2025-02-12 00:39:36.460000 → 2025-02-21 14:35:42 → 2025-04-02 13:39:01.800000


## STEP 6 — Preprocessing & Metrics utilities (imbalance-aware)

**Goal**  
Provide (1) a reusable preprocessing pipeline for numeric/categorical features and (2) evaluation helpers tailored to class imbalance (PR-first mindset and threshold tuning).

**Why**  
Consistent preprocessing avoids data leaks and keeps baselines comparable. In rare-event AML, **PR-AUC** and top-K precision/recall are more decision-aligned than accuracy; explicit threshold tuning on the **validation** set produces operational cutoffs.

**What this cell does**  
- Defines `pre`: a `ColumnTransformer` that
  - scales numerics with `StandardScaler`  
  - one-hot encodes low-card categoricals with `OneHotEncoder(handle_unknown="ignore", sparse=False)`
- Defines metrics/utilities:
  - `pr_auc(y_true, y_score)`: Average Precision (area under PR curve)  
  - `precision_recall_at_k(y_true, y_score, k)`: Top-K alert quality (how many true AMLs in the first *k*)  
  - `tune_threshold(y_true, y_score, objective, target_recall)`: grid-search over score quantiles to optimize **F1** (default) or **precision**, with an optional minimum recall constraint

**How**  
1. Build `pre` using `num_cols` and `cat_cols` defined earlier.  
2. Keep `remainder="drop"` to ensure only vetted features enter the model.  
3. For threshold tuning, iterate over score quantiles (1%…99%) and track best score (F1 or precision); optionally enforce `recall ≥ target_recall`.

**Inputs / Dependencies**  
- `num_cols`, `cat_cols` from STEP 4.  
- `numpy`, `pandas`, and `scikit-learn` (`ColumnTransformer`, `StandardScaler`, `OneHotEncoder`, metrics).

**Outputs / Side-effects**  
- `pre`: ready to be composed with an estimator in a `Pipeline`.  
- Utility functions usable for validation and test evaluation.  
- No files written.

**Acceptance criteria**  
- `pre` fits on **Train only** without errors; `transform` works on Val/Test.  
- `OneHotEncoder` gracefully handles unseen categories (`handle_unknown="ignore"`).  
- `pr_auc` returns a float in `[0,1]`; `precision_recall_at_k` returns `(precision, recall, tp)` with `0 ≤ k ≤ n`.  
- `tune_threshold` returns a dict with keys: `thr`, `f1`, `precision`, `recall`.

**Risks & controls**  
- *Missing values*: `StandardScaler` and `OneHotEncoder` don’t impute. If NaNs exist, extend `pre` with imputers:  
  ```python
  from sklearn.impute import SimpleImputer
  pre = ColumnTransformer([
      ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                        ("sc", StandardScaler(with_mean=True))]), num_cols),
      ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse=False))]), cat_cols)
  ], remainder="drop")


In [7]:
# ==== CELL 6: Preprocessing & Metrics utilities (NaN-safe) ====
"""
- Impute numerics (median), categoricals (constant '__missing__')
- Version-safe OneHotEncoder (sparse_output vs sparse)
- Drop columns that are 100% NaN in TRAIN to avoid imputer errors
"""
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score, precision_recall_curve, confusion_matrix,
    precision_score, recall_score, f1_score, roc_auc_score
)
import numpy as np

# --- version-safe OHE ---
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)  # sklearn >=1.4
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)         # sklearn <1.4

# --- drop all-NaN features (based on TRAIN only) ---
all_nan_num = [c for c in num_cols if X_train[c].isna().all()]
all_nan_cat = [c for c in cat_cols if X_train[c].isna().all()]
if all_nan_num or all_nan_cat:
    print("Dropping all-NaN cols:", all_nan_num + all_nan_cat)
    num_cols = [c for c in num_cols if c not in all_nan_num]
    cat_cols = [c for c in cat_cols if c not in all_nan_cat]

# --- pipelines with imputers ---
num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),                    # robust for numerics
    ("scale",  StandardScaler(with_mean=True))
])
cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="__missing__")),  # safe even if all NaN in VAL/TEST
    ("ohe",    ohe)
])

pre = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
], remainder="drop")

# ---- metrics helpers ----
def pr_auc(y_true, y_score):
    return float(average_precision_score(y_true, y_score))

def precision_recall_at_k(y_true, y_score, k=100):
    order = np.argsort(-y_score)
    top = order[:min(k, len(order))]
    tp  = y_true[top].sum()
    prec = tp / len(top) if len(top) else 0.0
    rec  = tp / y_true.sum() if y_true.sum() else 0.0
    return float(prec), float(rec), int(tp)

def tune_threshold(y_true, y_score, objective="f1", target_recall=None):
    qs = np.unique(np.quantile(y_score, np.linspace(0.01, 0.99, 99)))
    best = {"thr":0.5, "f1":-1, "precision":0, "recall":0}
    for thr in qs:
        y_pred = (y_score >= thr).astype(int)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec  = recall_score(y_true, y_pred, zero_division=0)
        f1v  = f1_score(y_true, y_pred, zero_division=0)
        if target_recall is not None and rec < target_recall:
            continue
        score = f1v if objective=="f1" else prec
        if score > best[objective]:
            best = {"thr": float(thr), "f1": float(f1v), "precision": float(prec), "recall": float(rec)}
    return best


## STEP 7 — Baseline models (LogReg, RF, XGBoost if available)

**Goal**  
Train strong, interpretable baselines with class-imbalance handling and select the best model by **validation PR-AUC** (primary metric).

**Why**  
In AML (rare events), **PR-AUC** reflects alert quality under skew far better than accuracy. Locking a baseline early gives a stable reference for all later feature and model experiments.

**What this cell does**  
- Trains three baselines using the shared preprocessing `pre`:
  - **LogisticRegression** (`class_weight="balanced"`) for linear baseline + coefficients.  
  - **RandomForest** (`class_weight="balanced_subsample"`) for non-linear baseline and quick feature importances.  
  - **XGBoost** (if installed) with `scale_pos_weight = neg/pos` for calibrated skew handling.  
- Captures **validation scores** (`PR-AUC`) and stores trained pipelines + scores in `results` / `models`.  
- Prints a sorted leaderboard by **Val PR-AUC**.

**How**  
1. Compose each estimator inside a `Pipeline([("pre", pre), ("clf", ...)])` to avoid leakage and keep transforms identical.  
2. For XGBoost, compute `scale_pos_weight` from Train split to reflect operational skew.  
3. Evaluate on **Val** only; defer Test usage until the winner and threshold are fixed.

**Inputs / Dependencies**  
- From previous steps: `X_train`, `y_train`, `X_val`, `y_val`, `X_test`, `y_test`, `pre`, `RNG`.  
- Optional: `xgboost` for the third baseline.

**Outputs / Side-effects**  
- `results`: dict of model → Val PR-AUC (float).  
- `models`: dict of model → (fitted_pipeline, p_val, p_test).  
- Console: ranked leaderboard for quick selection.

**Acceptance criteria**  
- All pipelines fit without error on **Train** and score on **Val**.  
- `results` contains at least **LogisticRegression** and **RandomForest**.  
- Reported PR-AUC is **strictly from Val** (no peeking at Test).  
- XGBoost (if present) uses `scale_pos_weight ≥ 1` when positives are rare.

**Risks & controls**  
- *Overfitting RF/XGB*: Use moderate depth/trees; prefer early, conservative configs (e.g., `max_depth=5`, subsampling).  
- *Threshold leakage*: **Do not** tune thresholds on Test; tune on **Val**, then freeze.  
- *Dense OHE memory*: `sparse=False` can inflate memory; if large, consider sparse output in `pre`.  
- *Class prevalence drift*: Validate per-split positive rates; if Train is too sparse, consider `class_weight`, resampling, or focal losses (later experiments).

**Observability (quick checks)**  
After this cell, record in the experiment log: model hyperparams, Val PR-AUC, and notes on fit time. Example fields:  
`[timestamp | commit | data_ver | split scheme | model | Val PR-AUC | notes]`.

**Next step**  
Pick the **Val PR-AUC winner**, **tune the decision threshold on Val** (e.g., target recall floor), **freeze**, then evaluate final metrics on **Test** (PR-AUC primary, ROC-AUC secondary, plus precision/recall/F1, confusion matrix). Add lightweight explainability (coefficients or feature importances).


In [8]:
# ==== CELL 7: Baseline models (LogReg, RF, XGBoost if available) ====
"""
Purpose: Train competitive baselines with class-imbalance handling.
- LogisticRegression with class_weight='balanced'
- RandomForest with class_weight='balanced_subsample'
- XGBoost with scale_pos_weight (if installed)
"""
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Try XGBoost (optional)
HAS_XGB = False
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    pass

results = {}
models  = {}

# Logistic Regression
logreg = Pipeline([
    ("pre", pre),
    ("clf", LogisticRegression(
        penalty="l2", solver="liblinear", class_weight="balanced",
        random_state=RNG, max_iter=300
    ))
])
logreg.fit(X_train, y_train)
p_val_lr  = logreg.predict_proba(X_val)[:,1]
p_test_lr = logreg.predict_proba(X_test)[:,1]
results["LogisticRegression"] = pr_auc(y_val, p_val_lr)
models["LogisticRegression"]  = (logreg, p_val_lr, p_test_lr)

# Random Forest
rf = Pipeline([
    ("pre", pre),
    ("clf", RandomForestClassifier(
        n_estimators=400, min_samples_leaf=2, class_weight="balanced_subsample",
        n_jobs=-1, random_state=RNG
    ))
])
rf.fit(X_train, y_train)
p_val_rf  = rf.predict_proba(X_val)[:,1]
p_test_rf = rf.predict_proba(X_test)[:,1]
results["RandomForest"] = pr_auc(y_val, p_val_rf)
models["RandomForest"]  = (rf, p_val_rf, p_test_rf)

# XGBoost (optional but often strong on tabular)
if HAS_XGB:
    pos = y_train.sum()
    neg = len(y_train) - pos
    spw = (neg / pos) if pos > 0 else 1.0
    xgb = Pipeline([
        ("pre", pre),
        ("clf", XGBClassifier(
            n_estimators=600, max_depth=5, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
            tree_method="hist", scale_pos_weight=spw, eval_metric="logloss",
            n_jobs=-1, random_state=RNG
        ))
    ])
    xgb.fit(X_train, y_train)
    p_val_xgb  = xgb.predict_proba(X_val)[:,1]
    p_test_xgb = xgb.predict_proba(X_test)[:,1]
    results["XGBoost"] = pr_auc(y_val, p_val_xgb)
    models["XGBoost"]  = (xgb, p_val_xgb, p_test_xgb)

print("Validation PR-AUC:")
for k,v in sorted(results.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {k:20s}  PR-AUC={v:.4f}")


Validation PR-AUC:
  RandomForest          PR-AUC=1.0000
  XGBoost               PR-AUC=1.0000
  LogisticRegression    PR-AUC=0.9933


## STEP 8 — Threshold tuning (Val) & final evaluation (Test)

**Goal**  
Select the best baseline by **validation PR-AUC**, tune an operational decision threshold on **Val** (F1 by default or with a recall floor), then **freeze** that threshold and report final **Test** metrics.

**Why**  
In rare-event AML, calibration of the decision threshold matters as much as the model choice. Tuning on **Val** (not Test) prevents optimistic bias and yields an actionable operating point aligned with risk appetite (e.g., minimum recall).

**What this cell does**  
- Picks the validation winner from `results`.  
- Tunes a threshold on **Val** via `tune_threshold(...)` (F1 objective; optional `target_recall`).  
- Applies the frozen threshold to **Test** scores and reports:
  - **PR-AUC** (primary) and **ROC-AUC** (secondary) on Test  
  - Precision / Recall / F1 at the tuned threshold  
  - Confusion matrix `[TN FP; FN TP]`  
  - **Precision@K** (Top-50/100/250/500) to approximate alert-queue quality

**How**  
1. `best_name = argmax(results by PR-AUC)`; retrieve `(best_model, p_val_best, p_test_best)`.  
2. Tune threshold on Val scores: `best_thr = tune_threshold(y_val, p_val_best, objective="f1", target_recall=None)`.  
3. Freeze `thr = best_thr["thr"]`; generate Test predictions and compute metrics.

**Inputs / Dependencies**  
- `results`, `models` from STEP 7; validation & test scores.  
- `y_val`, `y_test`; helper functions `pr_auc`, `precision_recall_at_k`, `tune_threshold`.

**Outputs / Side-effects**  
- Printed selection, tuned threshold, and Test metrics.  
- No files written (can be extended to log artifacts).

**Acceptance criteria**  
- Winner chosen strictly by **Val PR-AUC** (no Test peeking).  
- Threshold tuned on **Val** only, then **fixed**.  
- Test metrics printed without errors and include PR-AUC, ROC-AUC, P/R/F1, confusion matrix, and Precision@K.  
- Metrics are numerically consistent (e.g., F1 coherent with P and R at the same threshold).

**Risks & controls**  
- *Threshold overfitting to Val*: Consider k-fold time-series CV for robustness in later iterations; keep this as a simple baseline.  
- *Metric misalignment*: If operations demand high recall, set `target_recall` (e.g., `0.80`) and optimize precision or F1 subject to that constraint.  
- *Class prevalence drift*: Revisit threshold if Test prevalence deviates sharply from Val; document drift and consider periodic recalibration in production.  
- *Score calibration*: If thresholds prove unstable, add probability calibration (e.g., `CalibratedClassifierCV`) in future experiments.

**Observability (optional)**
```python
# Quick metric bundle for your experiment log
log_row = {
    "model": best_name,
    "thr": round(thr, 4),
    "test_pr_auc": round(ap_test, 4),
    "test_roc_auc": round(roc_test, 4),
    "test_precision": round(prec_test, 4),
    "test_recall": round(rec_test, 4),
    "test_f1": round(f1_test, 4),
    "cm_TN": int(cm[0,0]), "cm_FP": int(cm[0,1]),
    "cm_FN": int(cm[1,0]), "cm_TP": int(cm[1,1]),
}
print(log_row)


In [9]:
# ==== CELL 8: Threshold tuning on Val and Test evaluation ====
"""
Purpose:
- Select best model by PR-AUC on validation.
- Tune decision threshold (F1 by default; you can target a minimum recall).
- Report Test metrics: PR-AUC, ROC-AUC, Precision/Recall/F1 at tuned threshold, Confusion Matrix, Precision@K.
"""
from sklearn.metrics import roc_auc_score

best_name = max(results.items(), key=lambda kv: kv[1])[0]
best_model, p_val_best, p_test_best = models[best_name]
print("Selected model:", best_name)

# Tune threshold (change to target_recall=0.80 if policy requires minimum recall)
best_thr = tune_threshold(y_val, p_val_best, objective="f1", target_recall=None)
thr = best_thr["thr"]
print("Best threshold (Val):", best_thr)

# Evaluate on Test
yhat_test = (p_test_best >= thr).astype(int)
ap_test   = pr_auc(y_test, p_test_best)
roc_test  = roc_auc_score(y_test, p_test_best)
prec_test = precision_score(y_test, yhat_test, zero_division=0)
rec_test  = recall_score(y_test, yhat_test, zero_division=0)
f1_test   = f1_score(y_test, yhat_test, zero_division=0)
cm        = confusion_matrix(y_test, yhat_test)

print("\n=== Test metrics ===")
print(f"PR-AUC={ap_test:.4f}  ROC-AUC={roc_test:.4f}  P={prec_test:.4f}  R={rec_test:.4f}  F1={f1_test:.4f}")
print("Confusion matrix [TN FP; FN TP]:\n", cm)

for k in [50, 100, 250, 500]:
    Pk, Rk, TPk = precision_recall_at_k(y_test, p_test_best, k=k)
    print(f"@Top-{k:>4}: Precision={Pk:.3f}  Recall={Rk:.3f}  TP={TPk}")


Selected model: RandomForest
Best threshold (Val): {'thr': 0.00248855828261014, 'f1': 0.21536351165980797, 'precision': 0.12067640276710223, 'recall': 1.0}

=== Test metrics ===
PR-AUC=1.0000  ROC-AUC=1.0000  P=0.1438  R=1.0000  F1=0.2514
Confusion matrix [TN FP; FN TP]:
 [[106722   1334]
 [     0    224]]
@Top-  50: Precision=1.000  Recall=0.223  TP=50
@Top- 100: Precision=1.000  Recall=0.446  TP=100
@Top- 250: Precision=0.896  Recall=1.000  TP=224
@Top- 500: Precision=0.448  Recall=1.000  TP=224


## STEP 9 — Explainability (feature importances; optional SHAP)

**Goal**  
Provide fast, model-aligned interpretability to sanity-check the top drivers behind alerts and validate that features make domain sense.

**Why**  
Interpretability reduces the risk of spurious correlations, helps detect leakage, and supports stakeholder trust (professor/coach, hiring reviewers). Quick importances guide the next iteration of feature engineering.

**What this cell does**  
- **Builds feature names** after fitting the preprocessing step on **Train** (safe for leakage).  
- **Tree importances**: For RF/XGB, prints top-N `feature_importances_`.  
- **Optional SHAP**: On a small Test sample, computes Tree SHAP and prints top features by mean |SHAP| (heavier; off by default).

**How**  
1. `pre.fit(X_train)` strictly for deriving post-OHE feature names.  
2. Collect names in order: numerics first, then OHE category names via `get_feature_names_out(cat_cols)`.  
3. If the selected model exposes `feature_importances_`, sort and print the Top-25.  
4. (Optional) Sample up to 200 rows, transform with `pre`, run `shap.TreeExplainer` on the fitted classifier, aggregate mean |SHAP|, and print the Top-25.

**Inputs / Dependencies**  
- Trained winner (`best_name`, `best_model`) and `pre`, `num_cols`, `cat_cols`, `X_train`, `X_test`.  
- Optional: `shap` package installed for SHAP explanations.

**Outputs / Side-effects**  
- Console tables of Top-N features (by native importance or mean |SHAP|).  
- No files written (can be extended to write CSV/PNG artifacts).

**Acceptance criteria**  
- `feat_names` length equals the transformed feature dimension (post-OHE).  
- For RF/XGB, non-empty ranked list of importances; names align with expected domain drivers.  
- SHAP (if enabled) runs without error on the sample and returns a coherent ranking.

**Risks & controls**  
- *Pipeline structure for OHE names*: If your categorical branch is a `Pipeline` (e.g., with imputers), `pre.named_transformers_["cat"]` is not the encoder itself. Control:  
  ```python
  cat_block = pre.named_transformers_["cat"]
  try:
      cat_ohe = cat_block.named_steps["ohe"]  # if 'cat' is a Pipeline
  except AttributeError:
      cat_ohe = cat_block                     # if 'cat' is directly OneHotEncoder
  feat_names = list(num_cols) + list(cat_ohe.get_feature_names_out(cat_cols))


In [17]:
# ==== CELL 9: Explainability (feature importances; SHAP optional) ====
"""
Purpose: Provide fast interpretability to sanity-check top drivers.
- Tree-based importances for RF/XGB (fast).
- SHAP optional (can be heavier; enable if needed).
"""
# Build feature names after preprocessing
pre.fit(X_train)
feat_names = []
if len(num_cols): feat_names += num_cols
if len(cat_cols):
    cat_ohe = pre.named_transformers_["cat"]
    feat_names += list(cat_ohe.get_feature_names_out(cat_cols))

def show_importances(model_pipeline, topn=25):
    try:
        clf = model_pipeline.named_steps["clf"]
        importances = getattr(clf, "feature_importances_", None)
        if importances is None:
            print("No native feature_importances_ available.")
            return
        order = np.argsort(-importances)[:topn]
        print(f"\nTop-{topn} feature importances:")
        for i in order:
            nm = feat_names[i] if i < len(feat_names) else f"f{i}"
            print(f"{nm:50s} {importances[i]:.6f}")
    except Exception as e:
        print("Importance display failed:", e)

if best_name in ("RandomForest","XGBoost"):
    show_importances(best_model, topn=25)

# SHAP (optional)
USE_SHAP = False
if USE_SHAP and best_name in ("RandomForest","XGBoost"):
    try:
        import shap
        Xs = X_test.sample(n=min(200, len(X_test)), random_state=RNG)
        Xs_tr = best_model.named_steps["pre"].transform(Xs)
        expl = shap.TreeExplainer(best_model.named_steps["clf"])
        sv = expl.shap_values(Xs_tr)
        mean_abs = np.mean(np.abs(sv if isinstance(sv, np.ndarray) else sv[1]), axis=0)
        order = np.argsort(-mean_abs)[:25]
        print("\nTop SHAP features (mean |SHAP|):")
        for i in order:
            nm = feat_names[i] if i < len(feat_names) else f"f{i}"
            print(f"{nm:50s} {mean_abs[i]:.6f}")
    except Exception as e:
        print("SHAP failed gracefully:", e)



Top-25 feature importances:
Transaction_Type_Money Laundering                  0.105416
Laundering_Type_No Laundering                      0.098198
Is_Self_Transfer                                   0.079987
in_avg_30d                                         0.063998
out_avg_30d                                        0.053373
Is_RoundTrip_72h                                   0.052142
To_End_Balance                                     0.049676
To_Initial_Balance                                 0.046310
out_avg_7d                                         0.041640
in_avg_7d                                          0.040657
From_MCC                                           0.033418
Amount_Paid                                        0.027278
To_MCC                                             0.025607
out_sum_7d                                         0.025107
From_End_Balance                                   0.024477
From_Initial_Balance                               0.020301
in_sum_30d 

## STEP 10 — Persist minimal artifacts & run log

**Goal**  
Write a lightweight, review-friendly results file capturing the selected baseline, tuned threshold, key **Test** metrics, and split sizes—so downstream notebooks/dashboards can consume a stable summary without loading the whole model.

**Why**  
A small, schema-stable CSV enables quick reporting, comparisons across runs, and portfolio review (professor/coach/future employers) without digging through notebook cells or large artifacts.

**What this cell does**  
- Builds a one-row dictionary with:
  - `model`, tuned `threshold`
  - Test metrics: `test_pr_auc`, `test_roc_auc`, `test_precision`, `test_recall`, `test_f1`
  - Split sizes: `rows_train`, `rows_val`, `rows_test`
- Writes it to `${BASE_PATH}/baseline_results_v1.csv`.

**How**  
1. Assemble the `out` dict from variables computed in prior steps.  
2. Persist to CSV (append/aggregate in later iterations if desired).  
3. Print the absolute path for quick verification.

**Inputs / Dependencies**  
- Variables from Steps 5–8: `best_name`, `thr`, `ap_test`, `roc_test`, `prec_test`, `rec_test`, `f1_test`, `X_train`, `X_val`, `X_test`.  
- `BASE_PATH` from Step 1; write permissions.

**Outputs / Side-effects**  
- File: `baseline_results_v1.csv` (one row per execution in the current form).  
- Console message with the written path.

**Acceptance criteria**  
- File exists at `${BASE_PATH}/baseline_results_v1.csv`.  
- Columns (schema) exactly:  
  `model, threshold, test_pr_auc, test_roc_auc, test_precision, test_recall, test_f1, rows_train, rows_val, rows_test`.  
- Numeric fields are finite (no NaN/inf).  
- `threshold ∈ [0,1]` and `0 ≤ metrics ≤ 1`.

**Risks & controls**  
- *Accidental overwrite*: If you want to keep a history, switch to **append** with a timestamp & commit hash (see below).  
- *Path issues*: Ensure the directory exists; create it if missing.  
- *Partial writes*: Use an atomic write pattern if runs are concurrent.

**Observability (optional hardening)**
```python
# Ensure directory + atomic write
import os, time, json
os.makedirs(os.path.dirname(out_df_path), exist_ok=True)
tmp_path = out_df_path + ".tmp"
pd.DataFrame([out]).to_csv(tmp_path, index=False)
os.replace(tmp_path, out_df_path)  # atomic on POSIX

# Optional: line-delimited JSON audit trail (append)
audit = {**out, "ts": pd.Timestamp.utcnow().isoformat(), "branch": os.getenv("GIT_BRANCH", "")}
with open(f"{BASE_PATH}/baseline_results_audit.jsonl", "a") as f:
    f.write(json.dumps(audit) + "\n")
print("Audit appended:", f"{BASE_PATH}/baseline_results_audit.jsonl")


In [20]:
# ==== CELL 10: Persist minimal artifacts & run log ====
"""
Purpose: Save light-weight outputs for downstream notebooks/dashboards.
- baseline_results_v1.csv with chosen model + tuned threshold and key test metrics.
"""
out = {
    "model": best_name,
    "threshold": float(thr),
    "test_pr_auc": float(ap_test),
    "test_roc_auc": float(roc_test),
    "test_precision": float(prec_test),
    "test_recall": float(rec_test),
    "test_f1": float(f1_test),
    "rows_train": int(len(X_train)), "rows_val": int(len(X_val)), "rows_test": int(len(X_test))
}
out_df_path = f"{BASE_PATH}/baseline_results_v1.csv"
pd.DataFrame([out]).to_csv(out_df_path, index=False)
print("Wrote:", out_df_path)

Wrote: /content/drive/MyDrive/Portfolio/AML/core_banking_and_money_laundering/baseline_results_v1.csv


# **Validation Plan — AML, Recall-First**


**Goal**  
Train, compare, and validate supervised classifiers on the IBM synthetic core-banking AML data. Optimize for **high Recall** (minimize false negatives) while respecting business guardrails on **Precision** (control alert volume). Produce transparent metrics, a tuned decision threshold, and model explainability artifacts.

**Why**  
In AML, missing a laundering case (FN) is far costlier than investigating an extra alert (FP). Since the IBM synthetic dataset provides complete ground-truth labelling, PR-focused selection and thresholding are appropriate and robust. *Primary selection metric:* **Average Precision (PR-AUC)**; *Operational KPI:* **Recall@Precision≥P** (or **Recall@AlertRate K%**).

**What**  
- Train Logistic Regression, Random Forest, and Gradient Boosting (XGBoost if available).
- Cross-validated **PR-AUC** and **ROC-AUC** (validation).
- Threshold tuning on validation to **maximize Recall** subject to **Precision ≥ P** (default P=0.20) and/or **Max FPR ≤ r**.
- Final test evaluation with Confusion Matrix, PR/ROC curves, and a summary table.
- Explainability: Permutation importance + SHAP summary plot (tree) / coefficients (logistic).

**How (high-level)**  
1) Load cleaned structured transfers; auto-detect label column; split train/val/test (stratified).  
2) Preprocess (numeric scaling, one-hot categoricals).  
3) Fit baselines; compute CV PR-AUC; tune threshold on validation.  
4) Refit (train+val), lock threshold, evaluate on test.  
5) Explainability artifacts for the recommended model.  
6) Persist results CSV + figures.

**Output**  
- `results/val_summary.csv`, `results/test_summary.csv`  
- `baseline_results_v1.csv` (model, threshold, key test metrics)  
- `fig_pr_curve.png`, `fig_roc_curve.png`, `fig_cm.png`, `fig_importance.png`, `fig_shap.png`  
- Saved model: `models/best_model.joblib`

**Result**  
A recall-optimized, documented baseline with clear trade-offs and explainability for AML operations.

### Setup & Imports

In [21]:
# ==== CELL 2: Setup & imports ====
# If XGBoost/SHAP aren't present, this is safe; they are optional.
import sys, os, json, math, itertools, warnings, gc
warnings.filterwarnings("ignore")

# Core
import numpy as np, pandas as pd

# Model stack
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_predict
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve, roc_curve,
    confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.calibration import CalibratedClassifierCV

# Plotting
import matplotlib.pyplot as plt

# Optional: XGBoost + SHAP
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False

# IO utils
from pathlib import Path
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paths
BASE_PATH = Path("./")  # adapt for Drive if needed, e.g., Path("/content/drive/MyDrive/Portfolio/AML/...")
DATA_CANDIDATES = [
    BASE_PATH / "xfers_structured_clean_v1.csv",
    BASE_PATH / "xfers_structured_v1.csv",
    BASE_PATH / "xfers_enriched_v3.csv",
    BASE_PATH / "data" / "xfers_structured_clean_v1.csv",
]

OUT_DIR = BASE_PATH / "results"
FIG_DIR = BASE_PATH / "figs"
MODEL_DIR = BASE_PATH / "models"
for d in [OUT_DIR, FIG_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("Setup complete. XGB_AVAILABLE:", XGB_AVAILABLE, "SHAP_AVAILABLE:", SHAP_AVAILABLE)


Setup complete. XGB_AVAILABLE: True SHAP_AVAILABLE: True


### Load data & detect label, split sets

In [22]:
# ==== CELL 3: Load dataset, detect label, split train/val/test ====

def _find_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

csv_path = _find_existing(DATA_CANDIDATES)
assert csv_path is not None, f"Please set DATA_CANDIDATES to your cleaned transfers CSV."
print("Using data:", csv_path)

df = pd.read_csv(csv_path, low_memory=False)

# Heuristic label detection (common AML names)
LABEL_CANDIDATES = [
    "Is_Money_Laundering","is_money_laundering","Money_Laundering","money_laundering",
    "is_fraud","Fraud","Label","target","ml_flag","laundering_flag"
]
label_col = None
for c in LABEL_CANDIDATES:
    if c in df.columns:
        label_col = c
        break

if label_col is None:
    # fallback: pick a binary-looking column with plausible positive rate
    candidates = []
    for c in df.columns:
        s = df[c]
        if s.dropna().nunique() == 2:
            pos_rate = s.mean() if s.dtype != "O" else None
            try:
                if pos_rate is not None and 0.001 <= pos_rate <= 0.4:
                    candidates.append((c, pos_rate))
            except Exception:
                pass
    candidates = sorted(candidates, key=lambda x: abs(0.05 - x[1]))  # prefer ~5% positives
    assert candidates, "Could not detect label column. Please set LABEL_CANDIDATES or specify manually."
    label_col = candidates[0][0]
    print("Heuristic label chosen:", label_col)

y = df[label_col].astype(int).values

# Optional columns to drop (IDs, timestamps, raw text, unique keys)
EXCLUDE_COLS = {label_col, "Transaction_ID","From_Account","To_Account","From_Party_ID","To_Party_ID",
                "Fraudster_ID","Raw_Text","Note","Timestamp","Date","Datetime","dt"}

X = df.drop(columns=[c for c in EXCLUDE_COLS if c in df.columns]).copy()

# Identify types
num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in X.columns if c not in num_cols]

# Train/Val/Test split (60/20/20)
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.40, stratify=y, random_state=RANDOM_STATE)
X_val,   X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RANDOM_STATE)

print(f"Shapes -> Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print("Positive rate (train/val/test):", y_train.mean(), y_val.mean(), y_test.mean())


AssertionError: Please set DATA_CANDIDATES to your cleaned transfers CSV.

## Metric Strategy (Recall-first)

**Primary selection metric:**  
- **Average Precision (PR-AUC)** — better reflects performance under class imbalance and aligns with Recall–Precision trade-offs.

**Operational guardrails:**  
- **Recall@Precision ≥ P (default P=0.20)** — find the highest Recall while keeping Precision at or above an agreed minimum to limit false alerts.  
- Alternative: **Recall@MaxFPR ≤ r** — cap false-positive rate r (e.g., 5‰–2%).

**Secondary metrics:**  
- **ROC-AUC** (global ranking quality)
- **F1** (balanced Precision–Recall), **Confusion Matrix** at the tuned threshold

**Why:**  
In AML, the cost of **FN** is high. The IBM synthetic AML set provides fully accurate labels, making threshold tuning on validation reliable and transparent (good setting for PR-first selection).  


### Preprocess + model zoo + CV PR-AUC

In [ ]:
# ==== CELL 5: Preprocessing + model definitions + CV PR-AUC ====

# OneHotEncoder compatibility across sklearn versions
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

pre = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(with_mean=False), num_cols),
        ("cat", ohe, cat_cols)
    ],
    remainder="drop"
)

# Baselines
models = {
    "LogisticR": LogisticRegression(
        max_iter=1000, solver="saga", class_weight="balanced", n_jobs=None, penalty="l2", C=1.0, random_state=RANDOM_STATE
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=400, max_depth=None, min_samples_leaf=2, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE
    ),
}

if XGB_AVAILABLE:
    # Robust starting point; we’ll set scale_pos_weight on the fly in a pipeline step
    models["XGBoost"] = xgb.XGBClassifier(
        n_estimators=600, max_depth=6, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        tree_method="hist", eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1
    )

def evaluate_cv_ap(model_name, estimator):
    pipe = Pipeline([("pre", pre), ("clf", estimator)])
    # Out-of-fold probabilities for PR-AUC
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    oof_proba = cross_val_predict(pipe, X_train, y_train, cv=skf, method="predict_proba", n_jobs=-1)[:, 1]
    ap = average_precision_score(y_train, oof_proba)
    roc = roc_auc_score(y_train, oof_proba)
    return ap, roc

val_rows = []
for name, est in models.items():
    ap, roc = evaluate_cv_ap(name, est)
    val_rows.append({"model": name, "cv_pr_auc": ap, "cv_roc_auc": roc})
    print(f"{name}: CV PR-AUC={ap:.4f} | ROC-AUC={roc:.4f}")

val_summary = pd.DataFrame(val_rows).sort_values("cv_pr_auc", ascending=False).reset_index(drop=True)
display(val_summary)
val_summary.to_csv(OUT_DIR / "val_summary.csv", index=False)
print("Wrote:", OUT_DIR / "val_summary.csv")


### Fit, calibrate optional, threshold tuning on validation

In [ ]:
# ==== CELL 6: Fit top models, tune threshold on validation ====

PRECISION_FLOOR = 0.20  # adjust with AML ops
MAX_FPR = None          # e.g., 0.02 to cap false positive rate (optional)

def fit_pipe(name, est, X_tr, y_tr):
    pipe = Pipeline([("pre", pre), ("clf", est)])
    # Optional: set scale_pos_weight for XGBoost based on class ratio in training
    if XGB_AVAILABLE and isinstance(est, xgb.XGBClassifier):
        pos = y_tr.sum()
        neg = len(y_tr) - pos
        est.set_params(scale_pos_weight=(neg / max(pos, 1)))
    pipe.fit(X_tr, y_tr)
    return pipe

def best_threshold_by_precision(y_true, proba, precision_floor=0.20, max_fpr=None):
    prec, rec, thr = precision_recall_curve(y_true, proba)
    # Map thresholds -> (precision, recall)
    candidates = []
    # precision_recall_curve returns thr len = len(prec)-1
    for i, t in enumerate(thr):
        p, r = prec[i+1], rec[i+1]
        if p >= precision_floor:
            if max_fpr is not None:
                # compute fpr at this threshold
                y_hat = (proba >= t).astype(int)
                tn, fp, fn, tp = confusion_matrix(y_true, y_hat).ravel()
                fpr = fp / (fp + tn + 1e-9)
                if fpr > max_fpr:
                    continue
            candidates.append((r, t, p))
    if not candidates:
        # fall back to threshold giving max F1
        f1s = []
        for i, t in enumerate(thr):
            p, r = prec[i+1], rec[i+1]
            f1 = 2*p*r/(p+r+1e-12)
            f1s.append((f1, t, p, r))
        f1, t, p, r = max(f1s, key=lambda x:x[0])
        return t, {"precision": p, "recall": r, "mode": "F1_fallback"}
    r, t, p = max(candidates, key=lambda x: x[0])
    return t, {"precision": p, "recall": r, "mode": "precision_floor"}

fitted = {}
thr_info = {}
for name, est in models.items():
    m = fit_pipe(name, est, X_train, y_train)
    fitted[name] = m
    val_proba = m.predict_proba(X_val)[:,1]
    t, info = best_threshold_by_precision(y_val, val_proba, PRECISION_FLOOR, MAX_FPR)
    thr_info[name] = {"threshold": float(t), **info}
    print(f"{name}: tuned thr={t:.4f} | recall@P≥{PRECISION_FLOOR:.2f}={info['recall']:.3f} | precision={info['precision']:.3f} | mode={info['mode']}")

thr_df = pd.DataFrame([{"model":k, **v} for k,v in thr_info.items()]).sort_values("recall", ascending=False)
display(thr_df)


### Final test evaluation + plots + summary

In [ ]:
# ==== CELL 7: Evaluate on TEST (confusion matrix, PR/ROC, summary) ====

def plot_pr_curve(y_true, proba, thr=None, path=None, title="Precision-Recall Curve"):
    prec, rec, thr_grid = precision_recall_curve(y_true, proba)
    ap = average_precision_score(y_true, proba)
    plt.figure(figsize=(6,5))
    plt.plot(rec, prec, lw=2)
    plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(f"{title} (AP={ap:.3f})")
    if thr is not None:
        # Mark the tuned operating point
        from bisect import bisect_left
        idx = np.argmin(np.abs(thr_grid - thr))
        plt.scatter(rec[idx+1], prec[idx+1])
        plt.annotate(f"thr={thr:.3f}", (rec[idx+1], prec[idx+1]))
    if path: plt.savefig(path, bbox_inches="tight", dpi=150)
    plt.show()

def plot_roc_curve(y_true, proba, path=None, title="ROC Curve"):
    fpr, tpr, _ = roc_curve(y_true, proba)
    auc = roc_auc_score(y_true, proba)
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, lw=2)
    plt.plot([0,1],[0,1],'--', lw=1)
    plt.xlabel("FPR"); plt.ylabel("TPR (Recall)"); plt.title(f"{title} (ROC-AUC={auc:.3f})")
    if path: plt.savefig(path, bbox_inches="tight", dpi=150)
    plt.show()

test_rows = []
for name, m in fitted.items():
    thr = thr_info[name]["threshold"]
    proba = m.predict_proba(X_test)[:,1]
    yhat = (proba >= thr).astype(int)

    ap = average_precision_score(y_test, proba)
    roc = roc_auc_score(y_test, proba)
    cm = confusion_matrix(y_test, yhat)
    tn, fp, fn, tp = cm.ravel()
    prec = tp / (tp + fp + 1e-12)
    rec  = tp / (tp + fn + 1e-12)
    f1   = 2*prec*rec/(prec+rec+1e-12)
    alert_rate = (yhat.mean())

    test_rows.append({
        "model": name, "threshold": float(thr),
        "test_pr_auc": ap, "test_roc_auc": roc,
        "test_precision": prec, "test_recall": rec, "test_f1": f1,
        "test_alert_rate": alert_rate,
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)
    })

    # Plots (saved)
    plot_pr_curve(y_test, proba, thr=thr, path=FIG_DIR/f"fig_pr_{name}.png", title=f"PR — {name}")
    plot_roc_curve(y_test, proba, path=FIG_DIR/f"fig_roc_{name}.png", title=f"ROC — {name}")

test_summary = pd.DataFrame(test_rows).sort_values(["test_recall","test_pr_auc"], ascending=False).reset_index(drop=True)
display(test_summary)

test_summary.to_csv(OUT_DIR / "test_summary.csv", index=False)
print("Wrote:", OUT_DIR / "test_summary.csv")


## Model Selection & Recommendation

**Selection rule:**  
1) Rank by **Recall@Precision≥P** on validation (tuned threshold).  
2) Tie-break by **PR-AUC (AP)**.  
3) Then consider **stability** (CV dispersion) and **complexity/operational ease** (training time, serving cost, calibration needs).

**Recommendation template:**  
- If a tree-ensemble (e.g., XGBoost/RandomForest) beats LogisticR materially on **PR-AUC** and **Recall**, recommend the tree-ensemble (calibrated if needed) as **Best Model**.  
- If performance is close (ΔAP ≤ 0.01 and ΔRecall ≤ 0.01), prefer **LogisticR** for transparency, speed, and ease of governance.

We will print the auto-selected winner next and persist minimal artifacts.


In [ ]:
# ==== CELL 9: Auto-pick best model & persist run log ====
# Choose by Recall first, then AP
best_row = test_summary.sort_values(["test_recall","test_pr_auc"], ascending=False).iloc[0]
best_name = best_row["model"]
best_model = fitted[best_name]
thr = float(best_row["threshold"])

# Save model
best_path = MODEL_DIR / f"best_model_{best_name}.joblib"
joblib.dump({"pipeline": best_model, "threshold": thr, "label_col": label_col,
             "num_cols": num_cols, "cat_cols": cat_cols}, best_path)

# Minimal run log
out = {
    "model": best_name,
    "threshold": float(thr),
    "test_pr_auc": float(best_row["test_pr_auc"]),
    "test_roc_auc": float(best_row["test_roc_auc"]),
    "test_precision": float(best_row["test_precision"]),
    "test_recall": float(best_row["test_recall"]),
    "test_f1": float(best_row["test_f1"]),
    "test_alert_rate": float(best_row["test_alert_rate"]),
    "rows_train": int(len(X_train)), "rows_val": int(len(X_val)), "rows_test": int(len(X_test))
}
out_df_path = BASE_PATH / "baseline_results_v1.csv"
pd.DataFrame([out]).to_csv(out_df_path, index=False)
print("Best:", best_name)
print("Model saved:", best_path)
print("Run log:", out_df_path)
display(pd.DataFrame([out]))


### Eplainability: feature importance + SHAP/coefs

## Confusion Matrix @ Tuned Threshold

**Goal**  
Visualize true/false positives/negatives at the **validated operating threshold** (Recall-first with Precision guardrails) for the *selected best model*.

**Why**  
AML cares about **false negatives** (missed suspicious cases). The matrix at the tuned threshold shows the concrete trade-off between **TP**, **FP**, **TN**, **FN**.

**What**  
- Compute predictions on **TEST** using the tuned threshold.  
- Plot **Confusion Matrix** (raw counts) and a **normalized** version.

**How**  
Use the saved pipeline (`best_model`) and threshold (`thr` or `thr_info[best_name]`) to binarize probabilities and render 2 figures.

**Output**  
- `figs/fig_cm_<best_model>.png`  
- `figs/fig_cm_norm_<best_model>.png`  
- Printed `classification_report`

**Result**  
Clear view of errors and hit rate at the chosen operating point.


### Confusion Matrix

In [ ]:
# ==== CELL 12: Confusion Matrix @ tuned threshold ====
from sklearn.metrics import confusion_matrix, classification_report

# Safety: re-derive threshold if 'thr' doesn't exist
best_thr = float(thr) if "thr" in globals() else float(thr_info[best_name]["threshold"])

# Probabilities & hard predictions (TEST)
proba_best = best_model.predict_proba(X_test)[:, 1]
yhat_best = (proba_best >= best_thr).astype(int)

cm = confusion_matrix(y_test, yhat_best)
tn, fp, fn, tp = cm.ravel()

print(f"Best model: {best_name} | thr={best_thr:.4f}")
print("Confusion Matrix [TN FP; FN TP]:")
print(cm)
print("\nClassification report (TEST):")
print(classification_report(y_test, yhat_best, digits=3))

# Basic matplotlib heatmap (no seaborn)
import numpy as np
import matplotlib.pyplot as plt

def plot_confusion_matrix(cm, labels=("0","1"), normalize=False, title="Confusion Matrix", path=None):
    mat = cm.astype(float)
    if normalize:
        mat = mat / mat.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(5,4))
    im = ax.imshow(mat, interpolation="nearest")
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks(np.arange(len(labels))); ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels); ax.set_yticklabels(labels)
    fmt = ".2f" if normalize else "d"
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, format(mat[i, j], fmt), ha="center", va="center")
    fig.tight_layout()
    if path:
        plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()

labels = ["Non-ML", "ML"]
plot_confusion_matrix(cm, labels=labels, normalize=False,
                      title=f"Confusion Matrix — {best_name}",
                      path=FIG_DIR / f"fig_cm_{best_name}.png")

plot_confusion_matrix(cm, labels=labels, normalize=True,
                      title=f"Confusion Matrix (Normalized) — {best_name}",
                      path=FIG_DIR / f"fig_cm_norm_{best_name}.png")


## Probability Calibration — Reliability Curve

**Goal**  
Check whether predicted probabilities are **well-calibrated** (do 0.2 scores occur ≈20% positive, etc.?).

**Why**  
Operations, case triage, and threshold setting benefit from **reliable probabilities**. Poor calibration can distort alert volumes or risk scores.

**What**  
- Compute **reliability curve** (calibration curve) and **Brier score** on **TEST**.  
- Plot probability histogram to assess sharpness.

**How**  
Use `calibration_curve` with quantile bins (robust on imbalance). No re-fitting; just evaluate the selected model.

**Output**  
- `figs/fig_reliability_<best_model>.png`  
- `figs/fig_prob_hist_<best_model>.png`  
- Printed Brier score

**Result**  
Quick visual + numeric signal on calibration quality. If miscalibrated, consider a calibrated wrapper (isotonic/sigmoid) as a follow-up.


### Reliabity curve + Brier score + histogram

In [ ]:
# ==== CELL 14: Reliability curve + Brier score (TEST) ====
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

# Use TEST probabilities from the best model
proba_best = best_model.predict_proba(X_test)[:, 1]

# Reliability (quantile bins handle imbalance better)
prob_true, prob_pred = calibration_curve(y_test, proba_best, n_bins=10, strategy="quantile")

# Plot reliability curve (diagonal = perfect calibration)
plt.figure(figsize=(5,5))
plt.plot([0,1], [0,1], "--")
plt.plot(prob_pred, prob_true, marker="o")
plt.xlabel("Mean predicted probability")
plt.ylabel("Fraction of positives")
plt.title(f"Reliability curve — {best_name}")
plt.tight_layout()
plt.savefig(FIG_DIR / f"fig_reliability_{best_name}.png", dpi=150, bbox_inches="tight")
plt.show()

# Probability histogram (sharpness)
plt.figure(figsize=(5,4))
plt.hist(proba_best, bins=20)
plt.xlabel("Predicted probability")
plt.ylabel("Count")
plt.title(f"Score distribution — {best_name} (TEST)")
plt.tight_layout()
plt.savefig(FIG_DIR / f"fig_prob_hist_{best_name}.png", dpi=150, bbox_inches="tight")
plt.show()

# Brier score (lower is better)
brier = brier_score_loss(y_test, proba_best)
print(f"Brier score (TEST): {brier:.6f}")


## Optional: Calibrate the Best Model (Sigmoid or Isotonic)

**Goal**  
Wrap the best classifier with **probability calibration** using validation data, then re-evaluate on TEST.

**Why**  
If the reliability curve deviates strongly from the diagonal or Brier score is high, **Platt scaling** (sigmoid) or **isotonic** calibration can improve probability quality.

**What**  
- Refit a `CalibratedClassifierCV` on **train+val**.  
- Re-tune threshold on **VAL**.  
- Re-evaluate on **TEST** and compare.

**Output**  
- `models/best_model_calibrated.joblib`  
- Updated figures & summary row.

**Result**  
Better-calibrated scores that support threshold governance and case triage.


### Calibrated model + re-evaluation

In [ ]:
# ==== CELL 16: Fit calibrated wrapper (train+val), re-tune threshold, test ====
# Join TRAIN and VAL for final calibrated fit
X_trv = pd.concat([X_train, X_val], axis=0)
y_trv = np.concatenate([y_train, y_val])

base = fitted[best_name]  # existing pipeline(pre)->clf
preproc = base.named_steps["pre"]
clf_base = base.named_steps["clf"]

# Build a new pipeline: preproc -> calibrated(clf_base)
cal_method = "sigmoid"   # or "isotonic" (needs more data)
cal_cv = 3               # internal CV for calibration mapping
cal_clf = CalibratedClassifierCV(base_estimator=clf_base, cv=cal_cv, method=cal_method)

from sklearn.pipeline import Pipeline
cal_pipe = Pipeline([("pre", preproc), ("clf", cal_clf)])
cal_pipe.fit(X_trv, y_trv)

# Threshold tuning on VAL
val_proba_cal = cal_pipe.predict_proba(X_val)[:, 1]
cal_thr, cal_info = best_threshold_by_precision(y_val, val_proba_cal, PRECISION_FLOOR, MAX_FPR)

# Test evaluation
test_proba_cal = cal_pipe.predict_proba(X_test)[:, 1]
yhat_cal = (test_proba_cal >= cal_thr).astype(int)

ap_cal  = average_precision_score(y_test, test_proba_cal)
roc_cal = roc_auc_score(y_test, test_proba_cal)
cm_cal  = confusion_matrix(y_test, yhat_cal)
tn, fp, fn, tp = cm_cal.ravel()
prec_cal = tp / (tp + fp + 1e-12)
rec_cal  = tp / (tp + fn + 1e-12)
f1_cal   = 2*prec_cal*rec_cal/(prec_cal+rec_cal+1e-12)
brier_cal = brier_score_loss(y_test, test_proba_cal)

print(f"[CALIBRATED {best_name}] thr={cal_thr:.4f} mode={cal_info['mode']}")
print(f"TEST — AP={ap_cal:.4f} | ROC-AUC={roc_cal:.4f} | Precision={prec_cal:.4f} | Recall={rec_cal:.4f} | F1={f1_cal:.4f} | Brier={brier_cal:.6f}")
print("Confusion Matrix (TEST):")
print(cm_cal)

# Save calibrated model
cal_path = MODEL_DIR / f"best_model_{best_name}_calibrated.joblib"
joblib.dump({"pipeline": cal_pipe, "threshold": float(cal_thr), "label_col": label_col,
             "num_cols": num_cols, "cat_cols": cat_cols, "calibration": cal_method}, cal_path)
print("Saved calibrated model:", cal_path)
